# Instant Gratification: Unsupervised-First Approach

## Objectives

In this notebook, you will:

1. Understand why a simple supervised model fails on this competition
2. Discover hidden group structure in the data using clustering (KMeans and Gaussian Mixture Models)
3. Apply PCA separately within each cluster to reduce dimensionality and improve signal
4. Train specialized models (QDA, Logistic Regression, LightGBM) per cluster instead of one global model
5. Use soft clustering (GMM) for smoother decision boundaries
6. Apply pseudo-labeling to leverage confident predictions on test data
7. Ensemble multiple models and cluster configurations for final predictions
8. Generate a Kaggle submission

## Why Standard Supervised Learning Fails

The Instant Gratification dataset is unusual: it appears to be synthetic data generated from **multiple hidden distributions** rather than a single distribution. Standard machine learning assumes all data comes from one process, but here the data contains distinct subspaces or clusters.

### Problems with a single global model:

- **Data heterogeneity**: The target class distribution or feature relationships differ significantly between clusters
- **Feature entanglement**: Global PCA mixes signals from different subspaces, reducing interpretability
- **Decision boundary issues**: A single classifier may struggle at cluster boundaries
- **Overfitting to noise**: Without recognizing cluster structure, the model memorizes noise across all subspaces

### What we'll do instead:

| Problem | Solution | Approach |
|---------|----------|----------|
| Data from multiple hidden distributions | Detect subgroups first | **Unsupervised clustering (KMeans, GMM)** |
| Dimensionality too high for interpretability | Reduce per-cluster | **Apply PCA within each cluster (not globally)** |
| One model struggles at boundaries | Multiple specialized models | **Train separate classifier per cluster** |
| Hard cluster assignments lose information | Soft assignments with confidence | **Use GMM probabilistic memberships** |
| Single model has blind spots | Combine diverse approaches | **Ensemble different models and seeds** |

---

## TODO Checklist

- [ ] Task 1: Import libraries
- [ ] Task 2: Load and explore the dataset
- [ ] Task 3: Standardize features
- [ ] Task 4: Determine optimal number of clusters using elbow method and silhouette analysis
- [ ] Task 5: Perform global clustering with KMeans
- [ ] Task 6: Apply PCA separately within each cluster
- [ ] Task 7: Train QDA classifier per cluster
- [ ] Task 8: Train Logistic Regression classifier per cluster
- [ ] Task 9: Compare per-cluster models on validation data
- [ ] Task 10: Implement Gaussian Mixture Model (soft clustering)
- [ ] Task 11: Apply pseudo-labeling to improve predictions
- [ ] Task 12: Ensemble multiple models
- [ ] Task 13: Generate Kaggle submission
- [ ] Task 14: Summary and reflection

## Instructions

Follow each task step-by-step. Fill in the TODO sections with your code. This approach will reveal the hidden structure and significantly improve your score over a naive supervised model.

---
## Why Clustering First?

### The Unsupervised-First Paradigm

Most machine learning courses teach: **data → preprocess → supervised model → predict**.

But for heterogeneous data, this is backwards. We should:

1. **Discover the structure** (clustering, dimensionality reduction)
2. **Adapt to each substructure** (local preprocessing, local models)
3. **Combine results** (ensemble at a higher level)

### Why PCA per cluster instead of global?

- **Global PCA** finds directions of maximum variance across the entire dataset. But if the dataset has 10 clusters, PCA might align with inter-cluster variance (which is noise for within-cluster prediction).
- **Per-cluster PCA** finds the principal components within each cluster's local subspace. This isolates the true signal in that subspace and filters out cluster-to-cluster noise.

### Example visualization:

Imagine data in 2D with two clusters (circle and square):

```
Global PCA:
  X ← captures the distance between clusters (noise for classification within a cluster)
  Y ← captures variation within clusters (signal)

Per-cluster PCA:
  Circle cluster:
    PC1 ← variation within the circle (signal)
    PC2 ← more subtle within-circle variation
  
  Square cluster:
    PC1 ← variation within the square (signal)
    PC2 ← more subtle within-square variation
```

Per-cluster PCA gives each cluster its own coordinate system optimized for its local structure.

---
## Task 1: Import Required Libraries

In [ ]:
# TODO: Import core libraries
# - import pandas as pd
# - import numpy as np
# - import matplotlib.pyplot as plt
# - import seaborn as sns
# - import warnings
# - warnings.filterwarnings('ignore')

# TODO: Import sklearn clustering and decomposition
# - from sklearn.cluster import KMeans
# - from sklearn.mixture import GaussianMixture
# - from sklearn.decomposition import PCA
# - from sklearn.preprocessing import StandardScaler

# TODO: Import sklearn models
# - from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
# - from sklearn.linear_model import LogisticRegression
# - from sklearn.ensemble import RandomForestClassifier
# - from lightgbm import LGBMClassifier

# TODO: Import sklearn evaluation metrics
# - from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
# - from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
# - from sklearn.metrics import silhouette_score

# TODO: Set visualization style
# - sns.set_style('whitegrid')
# - plt.rcParams['figure.figsize'] = (12, 6)

# TODO: Set random seed for reproducibility
# - np.random.seed(42)

# TODO: Print success message
# print("All libraries imported successfully!")

---
## Task 2: Load and Explore the Dataset

### About Instant Gratification

This is a binary classification problem. The dataset contains:
- **train.csv**: Training features and target labels
- **test.csv**: Test features (no labels, we predict these)
- **sample_submission.csv**: Format for submission

The target is typically binary (0 or 1). Our goal is to predict the probability of class 1.

In [ ]:
# TODO: Load the training data
# - X_train_full = pd.read_csv('train.csv')
# - Print shape: X_train_full.shape
# - Print first few rows: X_train_full.head()

# TODO: Check for the target column
# - List all columns: X_train_full.columns.tolist()
# - Usually the target is named 'target' or 'class' or similar
# - Identify it and store as: y_train_full = X_train_full['target'] (or whatever the name is)

# TODO: Drop the target column from features
# - Also drop 'id' or 'ID' columns if present (they're just indices)
# - X_train_full = X_train_full.drop(['target', 'id'], axis=1, errors='ignore')

# TODO: Check data types and missing values
# - Print: X_train_full.dtypes
# - Print: X_train_full.isnull().sum().sum() (total missing values)
# - All features should be numeric. If not, handle or drop categorical features.

# TODO: Basic statistics
# - Print: X_train_full.describe()
# - Print: f"Target distribution:\n{y_train_full.value_counts()}"
# - Print: f"Target proportions:\n{y_train_full.value_counts(normalize=True)}"

In [ ]:
# TODO: Load test data (for final submission, we won't look at labels)
# - X_test_full = pd.read_csv('test.csv')
# - Store test IDs for later: test_ids = X_test_full['id'].values (if id column exists)
# - Drop ID column: X_test_full = X_test_full.drop('id', axis=1, errors='ignore')
# - Print shape: X_test_full.shape

# TODO: Verify that X_test_full has the same columns as X_train_full
# - Check: X_test_full.columns.equals(X_train_full.columns)
# - If not, reorder or handle missing columns

# TODO: Combine train and test for preprocessing
# - X_all = pd.concat([X_train_full, X_test_full], axis=0, ignore_index=True)
# - Store the split point: train_size = len(X_train_full)
# - Print: f"Combined shape: {X_all.shape}. Train: {train_size}, Test: {len(X_test_full)}"

---
## Task 3: Standardize Features

### Why standardization matters for clustering

- **KMeans** uses Euclidean distance. If features have different scales (one feature ranges 0-1000, another 0-1), the larger-scale feature will dominate clustering.
- **Standardization** (z-score normalization) puts all features on the same scale: mean = 0, std = 1.

$$x_{standardized} = \frac{x - \mu}{\sigma}$$

After standardization, each feature contributes equally to distance calculations.

In [ ]:
# TODO: Handle missing values
# - Fill with median (or mean): X_all = X_all.fillna(X_all.median())
# - Verify no missing values: print(X_all.isnull().sum().sum())

# TODO: Standardize all features
# - scaler = StandardScaler()
# - X_all_scaled = pd.DataFrame(
#     scaler.fit_transform(X_all),
#     columns=X_all.columns
#   )
# - Print: X_all_scaled.describe() to verify mean ≈ 0, std ≈ 1

# TODO: Split back into train and test
# - X_train_scaled = X_all_scaled[:train_size]
# - X_test_scaled = X_all_scaled[train_size:]
# - Print: f"Train shape: {X_train_scaled.shape}, Test shape: {X_test_scaled.shape}"

---
## Task 4: Determine Optimal Number of Clusters

### How many clusters are there?

We don't know in advance. Common approaches:

1. **Elbow Method**: Plot inertia (within-cluster variance) vs. number of clusters. The "elbow" (where the curve flattens) suggests the optimal k.
2. **Silhouette Score**: Measures how well-separated clusters are. Higher is better. Range: -1 to 1.
3. **Domain knowledge**: For Instant Gratification, research suggests ~512 clusters.

We'll use all three to guide our choice.

In [ ]:
# TODO: Elbow method
# - inertias = []
# - silhouette_scores = []
# - K_range = range(2, 100, 10)  (test k=2, 12, 22, ..., 92; adjust range as needed)
# - For each k in K_range:
#     kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
#     kmeans.fit(X_train_scaled)
#     inertias.append(kmeans.inertia_)
#     score = silhouette_score(X_train_scaled, kmeans.labels_)
#     silhouette_scores.append(score)
#     print(f"k={k}: inertia={kmeans.inertia_:.2f}, silhouette={score:.3f}")

# TODO: Plot elbow curve
# - fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
# - ax1.plot(K_range, inertias, 'bo-')
# - ax1.set_xlabel('Number of Clusters (k)')
# - ax1.set_ylabel('Inertia')
# - ax1.set_title('Elbow Method')
# - ax1.grid()
# - ax2.plot(K_range, silhouette_scores, 'ro-')
# - ax2.set_xlabel('Number of Clusters (k)')
# - ax2.set_ylabel('Silhouette Score')
# - ax2.set_title('Silhouette Analysis')
# - ax2.grid()
# - plt.tight_layout()
# - plt.show()

# TODO: Choose optimal k
# - Based on the plots and domain knowledge, pick a k value
# - For Instant Gratification, try k around 512 or values that show good silhouette scores
# - optimal_k = ???  (fill in your choice)
# - print(f"\nChosen k={optimal_k}")

---
## Task 5: Perform Global Clustering with KMeans

### KMeans Algorithm

1. Randomly initialize k cluster centers
2. Assign each point to the nearest center (Euclidean distance)
3. Update centers as the mean of all points in each cluster
4. Repeat steps 2-3 until convergence

After clustering, each training sample belongs to exactly one cluster. We'll use this to group samples and train specialized models.

In [ ]:
# TODO: Fit KMeans on training data
# - kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
# - train_clusters = kmeans_model.fit_predict(X_train_scaled)
# - Print: f"Train clusters assigned: {len(np.unique(train_clusters))} unique clusters"

# TODO: Predict clusters for test data
# - test_clusters = kmeans_model.predict(X_test_scaled)
# - Print: f"Test clusters assigned: {len(np.unique(test_clusters))} unique clusters"

# TODO: Analyze cluster distribution
# - cluster_counts = pd.Series(train_clusters).value_counts().sort_index()
# - print(f"Cluster sizes (min, max, mean): {cluster_counts.min()}, {cluster_counts.max()}, {cluster_counts.mean():.1f}")
# - Visualize: plt.figure(figsize=(14, 4))
#   plt.subplot(1, 2, 1)
#   cluster_counts.plot(kind='bar')
#   plt.title('Cluster Size Distribution (Train)')
#   plt.xlabel('Cluster ID')
#   plt.ylabel('Count')
#   plt.tight_layout()
#   plt.show()

# TODO: Store results
# - Add clusters to train dataframe:
#   X_train_scaled['cluster'] = train_clusters
#   X_test_scaled['cluster'] = test_clusters

---
## Task 6: Apply PCA Separately Within Each Cluster

### Per-Cluster PCA Strategy

Instead of global PCA on all 250 features:
1. For each cluster, apply PCA to compress features
2. Each cluster gets its own PCA transformation (not a global one)
3. Reduce dimensionality to 40-60 components per cluster (experiment!)
4. This captures the signal within each subspace, removing inter-cluster noise

### Why 40-60 components?

- Too few (< 30): lose signal, underfitting
- Too many (> 150): don't reduce enough, risk overfitting
- 40-60: good balance, typically explains 85-95% of variance within each cluster

In [ ]:
# TODO: Choose number of PCA components
# - n_components = 50  (experiment with 40, 50, 60)
# - print(f"Using {n_components} PCA components per cluster")

# TODO: Store PCA transformers for each cluster
# - pca_models = {}  (dictionary: cluster_id -> PCA transformer)

# TODO: Transform training data with per-cluster PCA
# - X_train_pca_list = []
# - for cluster_id in np.unique(train_clusters):
#     mask = train_clusters == cluster_id
#     X_cluster = X_train_scaled.loc[mask, X_train_scaled.columns != 'cluster']
#     
#     pca = PCA(n_components=n_components, random_state=42)
#     X_cluster_pca = pca.fit_transform(X_cluster)
#     pca_models[cluster_id] = pca
#     
#     # Create a DataFrame for this cluster's PCA'd data
#     X_cluster_pca_df = pd.DataFrame(
#         X_cluster_pca,
#         columns=[f'PC{i}' for i in range(n_components)],
#         index=X_cluster.index
#     )
#     X_cluster_pca_df['cluster'] = cluster_id
#     X_train_pca_list.append(X_cluster_pca_df)
#     
#     # Print explained variance
#     explained_var = pca.explained_variance_ratio_.sum()
#     print(f"Cluster {cluster_id}: {explained_var:.2%} variance explained")

# TODO: Concatenate all clusters
# - X_train_pca = pd.concat(X_train_pca_list, axis=0)
# - print(f"X_train_pca shape: {X_train_pca.shape}")

In [ ]:
# TODO: Transform test data using the learned PCA models
# - X_test_pca_list = []
# - for cluster_id in np.unique(test_clusters):
#     if cluster_id not in pca_models:
#         # This cluster is in test but not train (rare). Skip or use a default PCA.
#         print(f"Warning: Cluster {cluster_id} in test but not in train. Skipping.")
#         continue
#     
#     mask = test_clusters == cluster_id
#     X_cluster = X_test_scaled.loc[mask, X_test_scaled.columns != 'cluster']
#     
#     pca = pca_models[cluster_id]
#     X_cluster_pca = pca.transform(X_cluster)
#     
#     X_cluster_pca_df = pd.DataFrame(
#         X_cluster_pca,
#         columns=[f'PC{i}' for i in range(n_components)],
#         index=X_cluster.index
#     )
#     X_cluster_pca_df['cluster'] = cluster_id
#     X_test_pca_list.append(X_cluster_pca_df)

# TODO: Concatenate test clusters
# - X_test_pca = pd.concat(X_test_pca_list, axis=0)
# - print(f"X_test_pca shape: {X_test_pca.shape}")

# TODO: Align indices (important for later)
# - X_train_pca = X_train_pca.sort_index()
# - X_test_pca = X_test_pca.sort_index()

---
## Task 7: Train QDA Classifier Per Cluster

### Quadratic Discriminant Analysis (QDA)

QDA is a powerful classifier for reduced-dimensional data (like after PCA):
- Assumes each class follows a Gaussian distribution
- Allows each class its own covariance matrix (hence "Quadratic")
- Works well on 40-60 dimensional PCA data

We'll train one QDA model **per cluster**, predicting the target label within that cluster's subspace.

In [ ]:
# TODO: Split data for validation
# - X_train_pca_train, X_train_pca_val, y_train_train, y_train_val = train_test_split(
#     X_train_pca, y_train_full.iloc[X_train_pca.index],
#     test_size=0.2,
#     random_state=42,
#     stratify=y_train_full.iloc[X_train_pca.index]
#   )
# - print(f"Train: {len(X_train_pca_train)}, Val: {len(X_train_pca_val)}")

# TODO: Train QDA for each cluster
# - qda_models = {}  (dictionary: cluster_id -> QDA model)
# - for cluster_id in sorted(np.unique(X_train_pca_train['cluster'])):
#     mask = X_train_pca_train['cluster'] == cluster_id
#     if mask.sum() < 10:  # Skip tiny clusters
#         print(f"Cluster {cluster_id}: too small ({mask.sum()}), skipping")
#         continue
#     
#     X_c = X_train_pca_train.loc[mask, [c for c in X_train_pca_train.columns if c.startswith('PC')]]
#     y_c = y_train_train.loc[mask]
#     
#     qda = QDA()
#     qda.fit(X_c, y_c)
#     qda_models[cluster_id] = qda
#     
#     print(f"Cluster {cluster_id}: QDA trained on {len(X_c)} samples")

# TODO: Evaluate on validation set
# - val_pred_qda = []
# - for idx, row in X_train_pca_val.iterrows():
#     cluster_id = int(row['cluster'])
#     if cluster_id not in qda_models:
#         val_pred_qda.append(0.5)  # Default to 0.5 if model not available
#         continue
#     
#     X_sample = row[[c for c in row.index if c.startswith('PC')]].values.reshape(1, -1)
#     pred = qda_models[cluster_id].predict_proba(X_sample)[0, 1]
#     val_pred_qda.append(pred)
# 
# - qda_auc = roc_auc_score(y_train_val, val_pred_qda)
# - print(f"QDA AUC on validation: {qda_auc:.4f}")

---
## Task 8: Train Logistic Regression Classifier Per Cluster

### Logistic Regression as Baseline

While QDA is powerful, Logistic Regression is simpler and often generalizes well:
- Linear decision boundary in the PCA space
- Regularized by default (L2 penalty)
- Fast to train and predict

We'll train one Logistic Regression per cluster and compare with QDA.

In [ ]:
# TODO: Train Logistic Regression for each cluster
# - lr_models = {}  (dictionary: cluster_id -> LogisticRegression model)
# - for cluster_id in sorted(np.unique(X_train_pca_train['cluster'])):
#     mask = X_train_pca_train['cluster'] == cluster_id
#     if mask.sum() < 10:
#         continue
#     
#     X_c = X_train_pca_train.loc[mask, [c for c in X_train_pca_train.columns if c.startswith('PC')]]
#     y_c = y_train_train.loc[mask]
#     
#     lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
#     lr.fit(X_c, y_c)
#     lr_models[cluster_id] = lr
#     
#     print(f"Cluster {cluster_id}: LogisticRegression trained on {len(X_c)} samples")

# TODO: Evaluate on validation set
# - val_pred_lr = []
# - for idx, row in X_train_pca_val.iterrows():
#     cluster_id = int(row['cluster'])
#     if cluster_id not in lr_models:
#         val_pred_lr.append(0.5)
#         continue
#     
#     X_sample = row[[c for c in row.index if c.startswith('PC')]].values.reshape(1, -1)
#     pred = lr_models[cluster_id].predict_proba(X_sample)[0, 1]
#     val_pred_lr.append(pred)
# 
# - lr_auc = roc_auc_score(y_train_val, val_pred_lr)
# - print(f"LogisticRegression AUC on validation: {lr_auc:.4f}")

# TODO: Compare QDA vs LR
# - if qda_auc > lr_auc:
#     print(f"QDA is better ({qda_auc:.4f} vs {lr_auc:.4f})")
# - else:
#     print(f"LR is better ({lr_auc:.4f} vs {qda_auc:.4f})")

---
## Task 9: Compare Per-Cluster Models on Validation Data

### Summary comparison

Generate a summary table comparing all approaches and select the best for final submission.

In [ ]:
# TODO: Create comparison table
# - results_df = pd.DataFrame({
#     'Model': ['QDA', 'LogisticRegression'],
#     'AUC': [qda_auc, lr_auc]
#   })
# - print("\nModel Comparison on Validation Set:")
# - print(results_df.to_string(index=False))

# TODO: Plot ROC curves
# - from sklearn.metrics import roc_curve
# - plt.figure(figsize=(8, 6))
# - fpr_qda, tpr_qda, _ = roc_curve(y_train_val, val_pred_qda)
# - fpr_lr, tpr_lr, _ = roc_curve(y_train_val, val_pred_lr)
# - plt.plot(fpr_qda, tpr_qda, label=f'QDA (AUC={qda_auc:.4f})')
# - plt.plot(fpr_lr, tpr_lr, label=f'LR (AUC={lr_auc:.4f})')
# - plt.plot([0, 1], [0, 1], 'k--', label='Random')
# - plt.xlabel('False Positive Rate')
# - plt.ylabel('True Positive Rate')
# - plt.title('ROC Curves: Per-Cluster Models')
# - plt.legend()
# - plt.grid()
# - plt.show()

# TODO: Select best model
# - best_model_type = 'QDA' if qda_auc > lr_auc else 'LogisticRegression'
# - best_models = qda_models if qda_auc > lr_auc else lr_models
# - print(f"\nSelected: {best_model_type}")

---
## Task 10: Implement Gaussian Mixture Model (Soft Clustering)

### Hard vs Soft Clustering

**Hard clustering (KMeans)**: Each point belongs to exactly one cluster. Binary assignment.

**Soft clustering (GMM)**: Each point has a probabilistic membership in each cluster. Sum of probabilities = 1.

Advantages of GMM:
- Smoother decision boundaries (no hard cluster jumps)
- Uncertainty quantified (membership probabilities)
- Can weight predictions by confidence

### Using GMM for ensemble

For each test point, we compute:
1. Membership probabilities across all clusters: $P(\text{cluster}_i | \text{point})$
2. Prediction from each cluster's model: $\hat{y}_i$
3. Weighted average: $\hat{y} = \sum_i P(\text{cluster}_i | \text{point}) \cdot \hat{y}_i$

In [ ]:
# TODO: Fit Gaussian Mixture Model
# - gmm_model = GaussianMixture(n_components=optimal_k, random_state=42)
# - gmm_model.fit(X_train_scaled.drop('cluster', axis=1))
# - print(f"GMM fitted with {optimal_k} components")

# TODO: Get soft cluster assignments for training data
# - train_gmm_probs = gmm_model.predict_proba(X_train_scaled.drop('cluster', axis=1))
# - print(f"Train GMM probabilities shape: {train_gmm_probs.shape}")

# TODO: Get soft cluster assignments for test data
# - test_gmm_probs = gmm_model.predict_proba(X_test_scaled.drop('cluster', axis=1))
# - print(f"Test GMM probabilities shape: {test_gmm_probs.shape}")

# TODO: Inspect GMM predictions
# - print(f"GMM probability example (first test sample):")
# - print(f"  Max prob: {test_gmm_probs[0].max():.4f}")
# - print(f"  Min prob: {test_gmm_probs[0].min():.4f}")
# - print(f"  Sum: {test_gmm_probs[0].sum():.4f} (should be 1.0)")

---
### Task 11: Apply Pseudo-Labeling to Improve Predictions

#### Pseudo-Labeling Strategy

1. Train initial model on labeled training data
2. Predict on test data with high confidence
3. Add the most confident test predictions as pseudo-labels
4. Retrain model on (original train + pseudo-labeled test)
5. Predict final submission

This leverages unlabeled test data to improve generalization. Key: only use **confident** predictions (e.g., probability > 0.95 or < 0.05).

In [ ]:
# TODO: Get predictions on test set from best model
# - test_pred_best = []
# - for idx, row in X_test_pca.iterrows():
#     cluster_id = int(row['cluster'])
#     if cluster_id not in best_models:
#         test_pred_best.append(0.5)
#         continue
#     
#     X_sample = row[[c for c in row.index if c.startswith('PC')]].values.reshape(1, -1)
#     model = best_models[cluster_id]
#     pred = model.predict_proba(X_sample)[0, 1]
#     test_pred_best.append(pred)

# TODO: Identify confident predictions
# - confidence_threshold = 0.95
# - confident_mask = (np.array(test_pred_best) > confidence_threshold) | (np.array(test_pred_best) < (1 - confidence_threshold))
# - confident_indices = np.where(confident_mask)[0]
# - print(f"Found {len(confident_indices)} confident predictions out of {len(test_pred_best)}")

# TODO: Create pseudo-labeled dataset
# - test_pred_best_array = np.array(test_pred_best)
# - pseudo_labels = (test_pred_best_array > 0.5).astype(int)
# - X_pseudo = X_test_pca.iloc[confident_indices].copy()
# - y_pseudo = pseudo_labels[confident_indices]
# - print(f"Pseudo-labeled set: {len(X_pseudo)} samples, {y_pseudo.sum()} positive")

# TODO: Retrain models with pseudo-labeled data
# - # Combine original train + pseudo-labeled test
# - X_train_pseudo = pd.concat([X_train_pca_train, X_pseudo], axis=0)
# - y_train_pseudo = pd.concat([y_train_train, pd.Series(y_pseudo, index=X_pseudo.index)], axis=0)
# - 
# - # Retrain best_models
# - best_models_v2 = {}
# - for cluster_id in sorted(np.unique(X_train_pseudo['cluster'])):
#     mask = X_train_pseudo['cluster'] == cluster_id
#     if mask.sum() < 10:
#         continue
#     
#     X_c = X_train_pseudo.loc[mask, [c for c in X_train_pseudo.columns if c.startswith('PC')]]
#     y_c = y_train_pseudo.loc[mask]
#     
#     model = type(best_models[cluster_id])()  # Create new instance of same model type
#     model.fit(X_c, y_c)
#     best_models_v2[cluster_id] = model
# - print("\nModels retrained with pseudo-labels")

---
## Task 12: Ensemble Multiple Models

### Ensemble Strategy

Different approaches make different errors. Combining them reduces variance:

1. **Per-cluster QDA** (hard clusters)
2. **Per-cluster LogisticRegression** (hard clusters)
3. **Per-cluster + pseudo-labeled** (improved hard clusters)
4. **GMM soft cluster predictions** (probabilistic clusters)

Average all predictions for robustness.

In [ ]:
# TODO: Helper function to get predictions from a model set
# def get_predictions_per_cluster(X_data, models, pca_models):
#     predictions = []
#     for idx, row in X_data.iterrows():
#         cluster_id = int(row['cluster'])
#         if cluster_id not in models:
#             predictions.append(0.5)
#             continue
#         
#         X_sample = row[[c for c in row.index if c.startswith('PC')]].values.reshape(1, -1)
#         pred = models[cluster_id].predict_proba(X_sample)[0, 1]
#         predictions.append(pred)
#     return np.array(predictions)

# TODO: Get predictions from all model variants
# - pred_qda = get_predictions_per_cluster(X_test_pca, qda_models, pca_models)
# - pred_lr = get_predictions_per_cluster(X_test_pca, lr_models, pca_models)
# - pred_pseudo = get_predictions_per_cluster(X_test_pca, best_models_v2, pca_models)

# TODO: Ensemble predictions (simple average)
# - ensemble_pred = (pred_qda + pred_lr + pred_pseudo) / 3
# - print(f"Ensemble prediction stats:")
# - print(f"  Min: {ensemble_pred.min():.4f}")
# - print(f"  Max: {ensemble_pred.max():.4f}")
# - print(f"  Mean: {ensemble_pred.mean():.4f}")
# - print(f"  Positive class ratio: {(ensemble_pred > 0.5).sum() / len(ensemble_pred):.2%}")

---
## Task 13: Generate Kaggle Submission

Format the predictions into a CSV file suitable for submission to Kaggle.

In [ ]:
# TODO: Create submission dataframe
# - submission = pd.DataFrame({
#     'id': test_ids,  # or range(len(ensemble_pred)) if no test_ids
#     'target': ensemble_pred
#   })
# - print(submission.head())
# - print(f"Submission shape: {submission.shape}")

# TODO: Save submission
# - submission.to_csv('submission.csv', index=False)
# - print("\nSubmission saved to submission.csv")

# TODO: Verify submission format
# - sample = pd.read_csv('sample_submission.csv')
# - print(f"\nSample submission shape: {sample.shape}")
# - print(f"Submission shape: {submission.shape}")
# - print(f"Match: {submission.shape == sample.shape}")

---
## Task 14: Summary and Reflection

### What We Built

A multi-stage unsupervised-then-supervised pipeline:

1. **Clustering** discovered hidden substructure (512 clusters)
2. **Per-cluster PCA** isolated signal within each subspace
3. **Per-cluster models** adapted to each subspace's characteristics
4. **Pseudo-labeling** leveraged confident test predictions
5. **Ensembling** combined multiple models for robustness

### Questions for Reflection

1. How did per-cluster PCA improve over global PCA?
2. Did QDA or LogisticRegression work better? Why?
3. How many clusters were optimal? Did you try different values?
4. Did pseudo-labeling improve your score? By how much?
5. What happens if you use GMM instead of KMeans?
6. How would you improve this approach further?

In [ ]:
# TODO: Write your reflection
# - Describe what you learned
# - Discuss your final score and how it compares to a naive supervised approach
# - List three things that worked well
# - List two things you'd do differently next time

reflection = """
## Summary

[Your reflection here]

### Final Kaggle Score

[Your submission score]

### Key Takeaways

1. 
2. 
3. 

### Future Improvements

1. 
2. 
"""

print(reflection)